# FinanceDataReader(fdr) 완전정복

한국·미국·글로벌 **금융 데이터를 코드 몇 줄로** 가져오는 라이브러리 학습 노트북입니다.

## 목차

| 장 | 주제 | 핵심 API |
|:--:|------|----------|
| 1 | [fdr이란?](#1장-fdr이란) | 설치, 3대 함수 |
| 2 | [DataReader 기초](#2장-DataReader-기초) | `fdr.DataReader('005930')` |
| 3 | [기간 지정](#3장-기간-지정) | `start`, `end` |
| 4 | [데이터 소스 지정](#4장-데이터-소스-지정) | `KRX:`, `NAVER:` |
| 5 | [글로벌 지수](#5장-글로벌-지수) | `KS11`, `IXIC`, `US500` |
| 6 | [미국 주식](#6장-미국-주식) | `AAPL`, `NASDAQ:AAPL` |
| 7 | [StockListing](#7장-StockListing) | `fdr.StockListing('KRX')` |
| 8 | [시장별 종목목록](#8장-시장별-종목목록) | KOSPI, NASDAQ, ETF |
| 9 | [SnapDataReader](#9장-SnapDataReader) | 지수목록, 스냅샷 |
| 10 | [지수 구성종목](#10장-지수-구성종목) | `KRX/INDEX/STOCK/1001` |
| 11 | [실전: 수익률 분석](#11장-실전-수익률-분석) | `pct_change`, 비교 |
| 12 | [실전: 종목 스크리닝](#12장-실전-종목-스크리닝) | 필터링 |
| — | [종합 퀴즈](#-종합-퀴즈) | 복습 |

## 학습 목표
- `DataReader`, `StockListing`, `SnapDataReader` 3가지 함수를 구분한다
- 한국/미국 주식·지수 데이터를 기간별로 조회한다
- 종목 목록에서 조건 필터링을 할 수 있다

## 사용 환경
```bash
pip install finance-datareader pandas
```

> **참고 (2026)**: KRX 일부 API는 [data.krx.co.kr](https://data.krx.co.kr) 로그인이 필요합니다.  
> `StockListing`, `DataReader`(주가), `KRX/INDEX/LIST` 등은 GitHub 캐시로 대부분 동작합니다.  
> 지수 구성종목은 `edupython/.env` 에 `KRX_ID`, `KRX_PW` 를 설정한 뒤 `pykrx` 로 조회할 수 있습니다.

## 1장. fdr이란?

**FinanceDataReader** = 금융 데이터 통합 리더기

| 함수 | 용도 | 반환 |
|------|------|------|
| `DataReader` | **시계열** (주가·지수·환율) | DataFrame (날짜 인덱스) |
| `StockListing` | **종목 목록** (스냅샷) | DataFrame |
| `SnapDataReader` | **특정 시점 스냅샷** (지수목록·구성종목 등) | DataFrame |

In [ ]:
import FinanceDataReader as fdr
import pandas as pd

print('fdr 버전:', getattr(fdr, '__version__', 'unknown'))
print('pandas 버전:', pd.__version__)

## 2장. DataReader 기초

가장 많이 쓰는 함수입니다. **종목코드만** 넘기면 과거~현재 주가를 가져옵니다.

```python
df = fdr.DataReader('005930')  # 삼성전자
```

반환 DataFrame 주요 컬럼: `Open`, `High`, `Low`, `Close`, `Volume`

In [ ]:
# 삼성전자 일봉 (최근 5행)
df = fdr.DataReader('005930')
print('shape:', df.shape)
print('컬럼:', list(df.columns))
print('기간:', df.index.min().date(), '~', df.index.max().date())
df.tail()

### 📝 실습 2-1
SK하이닉스(`000660`) 종가 `Close`의 최근 값과 전체 데이터 건수를 출력해 보세요.

In [ ]:
# 직접 작성
hynix = fdr.DataReader('000660')
print('최근 종가:', hynix['Close'].iloc[-1])
print('총 거래일:', len(hynix))

## 3장. 기간 지정

`start`, `end` 로 조회 구간을 제한합니다.

| 형식 | 예시 |
|------|------|
| 연도 | `'2025'` → 2025-01-01 ~ |
| 연-월 | `'2025-06'` |
| 연-월-일 | `'2025-01-01', '2025-12-31'` |

In [ ]:
# 2025년 삼성전자
df_2025 = fdr.DataReader('005930', '2025')
print(f'2025년 거래일: {len(df_2025)}일')
df_2025.head(3)

In [ ]:
# 시작·종료일 모두 지정
df_range = fdr.DataReader('005930', '2025-06-01', '2025-06-30')
print('6월 종가 통계:')
print(df_range['Close'].describe())

## 4장. 데이터 소스 지정

`소스:종목코드` 형식으로 데이터 출처를 명시할 수 있습니다.

| 형식 | 설명 |
|------|------|
| `'005930'` | 자동 선택 (한국→네이버, 미국→야후) |
| `'KRX:005930'` | KRX 공식 데이터 |
| `'NAVER:005930'` | 네이버 금융 |
| `'NASDAQ:AAPL'` | 나스닥 AAPL |

In [ ]:
# 동일 종목, 소스만 다르게 (최근 3일 비교)
auto = fdr.DataReader('005930', '2025-12-01').tail(3)['Close']
krx  = fdr.DataReader('KRX:005930', '2025-12-01').tail(3)['Close']

compare = pd.DataFrame({'자동': auto, 'KRX': krx})
print(compare)

## 5장. 글로벌 지수

지수 심볼만 넘기면 됩니다. (별도 `StockListing` 불필요)

| 심볼 | 지수 |
|------|------|
| `KS11` / `KOSPI` | 코스피 |
| `KQ11` / `KOSDAQ` | 코스닥 |
| `IXIC` | 나스닥 |
| `US500` / `S&P500` | S&P 500 |
| `DJI` | 다우존스 |

In [ ]:
# 코스피 vs 나스닥 — 최근 1개월
kospi = fdr.DataReader('KS11', '2025-12-01')
nasdaq = fdr.DataReader('IXIC', '2025-12-01')

print('코스피 최근 종가:', kospi['Close'].iloc[-1])
print('나스닥 최근 종가:', nasdaq['Close'].iloc[-1])

kospi.tail(3)

## 6장. 미국 주식

티커만 쓰거나 `거래소:티커` 형식을 사용합니다.

In [ ]:
# 애플 — 최근 3개월
aapl = fdr.DataReader('AAPL', '2025-10-01')
print('AAPL 최근 종가:', aapl['Close'].iloc[-1])
aapl.tail(3)

## 7장. StockListing

특정 시장의 **전체 종목 목록**을 한 번에 가져옵니다.

```python
df = fdr.StockListing('KRX')  # 한국거래소 전체
```

주요 컬럼: `Code`, `Name`, `Close`, `Changes`, `ChagesRatio`, `Volume`, `Marcap`

In [ ]:
krx = fdr.StockListing('KRX')
print(f'KRX 전체: {len(krx):,}종목')
print('컬럼:', list(krx.columns))
krx[['Code', 'Name', 'Market', 'Close', 'ChagesRatio', 'Marcap']].head()

## 8장. 시장별 종목목록

| market 인자 | 설명 |
|-------------|------|
| `'KRX'` | 한국거래소 전체 |
| `'KOSPI'` | 코스피 |
| `'KOSDAQ'` | 코스닥 |
| `'NASDAQ'` | 나스닥 |
| `'NYSE'` | 뉴욕 |
| `'S&P500'` | S&P 500 구성종목 |
| `'ETF/KR'` | 한국 ETF |

In [ ]:
# 코스피만
kospi_stocks = fdr.StockListing('KOSPI')
print(f'코스피: {len(kospi_stocks):,}종목')

# 한국 ETF
etf_kr = fdr.StockListing('ETF/KR')
print(f'한국 ETF: {len(etf_kr):,}종목')
etf_kr[['Symbol', 'Name', 'Price', 'ChangeRate']].head(3)

## 9장. SnapDataReader

**특정 시점의 스냅샷** 데이터를 조회합니다. (시계열이 아님)

```python
fdr.SnapDataReader('KRX/INDEX/LIST')  # KRX 전체 지수 목록
```

In [ ]:
# KRX 전체 지수 목록 (163개)
index_list = fdr.SnapDataReader('KRX/INDEX/LIST')
print(f'지수 {len(index_list)}개')
index_list.head(10)

In [ ]:
# 시장별 지수 개수
index_list.groupby('Market').size()

## 10장. 지수 구성종목

```python
fdr.SnapDataReader('KRX/INDEX/STOCK/1001')  # 코스피 구성종목
fdr.SnapDataReader('KRX/INDEX/STOCK/1028')  # 코스피200
fdr.SnapDataReader('KRX/INDEX/STOCK/1013')  # 전기전자 섹터
```

⚠️ **주의**: 2026년부터 KRX 로그인 없이는 `KRX/INDEX/STOCK/*` 조회가 실패할 수 있습니다.  
이 경우 `edupython/.env` 에 `KRX_ID`, `KRX_PW` 를 넣고 `pykrx` 로 조회하거나 GitHub 캐시를 사용합니다.

In [ ]:
# 코스피(1001) 구성종목 — 성공 시 종목 리스트 반환
try:
    kospi_members = fdr.SnapDataReader('KRX/INDEX/STOCK/1001')
    print(f'코스피 구성종목: {len(kospi_members)}개')
    kospi_members.head()
except Exception as e:
    print('조회 실패:', type(e).__name__)
    print('→ KRX 로그인 또는 캐시 방식 필요:', e)

## 11장. 실전: 수익률 분석

여러 종목의 **일간 수익률**을 비교해 봅니다.

In [ ]:
symbols = {'005930': '삼성전자', '000660': 'SK하이닉스', '035420': 'NAVER'}
start = '2025-01-01'

# 종가만 모아서 DataFrame 만들기
closes = pd.DataFrame()
for code, name in symbols.items():
    df = fdr.DataReader(code, start)
    closes[name] = df['Close']

# 일간 수익률 (%)
returns = closes.pct_change() * 100

print('=== 최근 5일 수익률(%) ===')
print(returns.tail().round(2))

print('\n=== 2025년 누적 수익률(%) ===')
cum = ((closes / closes.iloc[0]) - 1) * 100
for col in cum.columns:
    print(f'{col}: {cum[col].iloc[-1]:+.2f}%')

## 12장. 실전: 종목 스크리닝

`StockListing` 결과에서 **조건 필터**로 종목을 골라냅니다.

In [ ]:
df = fdr.StockListing('KOSPI')

# 등락률 컬럼명 (fdr 버전에 따라 ChagesRatio 또는 ChangeRate)
ratio_col = 'ChagesRatio' if 'ChagesRatio' in df.columns else 'ChangeRate'
df[ratio_col] = pd.to_numeric(df[ratio_col], errors='coerce')

# 조건: 등락률 3% 이상 상승
hot = df[df[ratio_col] >= 3].sort_values(ratio_col, ascending=False)
print(f'코스피 상승 3% 이상: {len(hot)}종목')
hot[['Code', 'Name', 'Close', ratio_col, 'Volume']].head(10)

In [ ]:
# 조건: 시가총액 상위 10개 중 오늘 상승 종목
df['Marcap'] = pd.to_numeric(df['Marcap'], errors='coerce')
top10 = df.nlargest(10, 'Marcap')
up_in_top10 = top10[top10[ratio_col] > 0]
print(f'시총 TOP10 중 상승: {len(up_in_top10)}종목')
up_in_top10[['Name', 'Close', ratio_col, 'Marcap']]

---

## 📝 종합 퀴즈

1. 삼성전자 **최근 1년** 주가를 가져오는 코드는?
2. 나스닥 **전체 종목 목록**을 가져오려면 `DataReader`와 `StockListing` 중 무엇을 쓸까?
3. 코스피 **지수 목록**(163개)을 가져오는 `SnapDataReader` 인자는?
4. `DataReader('KS11')`과 `DataReader('005930')`의 반환 데이터 공통점은?

<details>
<summary>정답 힌트</summary>

1. `fdr.DataReader('005930', '2025')` 또는 1년 전 날짜 지정
2. `StockListing('NASDAQ')`
3. `'KRX/INDEX/LIST'`
4. 둘 다 날짜 인덱스 + OHLCV 형태의 DataFrame

</details>